# 03 - Modelagem, IA e Sistemas Inteligentes

Notebook responsável pelas análises avançadas, modelagem estatística, inteligência artificial, sistemas de recomendação e experimentos de IA generativa aplicados aos dados de compras.

Objetivos:
- Market Basket Analysis
- Sistemas de Recomendação
- Embeddings de Produtos
- IA Generativa
- Insights Automáticos
- Modelagem Temporal

In [1]:
# !pip install -r requirements.txt

In [2]:
# IMPORTS
import time
from pathlib import Path
import pandas as pd
import warnings

from src.core.tictoc import tictoc
from src.core.config_loader import ConfigLoader
from src.core.paths import ROOT_DIR
from src.core.logger import setup_logger
from src.core.model import *

# CONFIGURAÇÃO
warnings.filterwarnings('ignore')
logger = setup_logger(__name__)
tic_geral = time.time()

config = ConfigLoader(
    "config/settings.yaml"
)

## 1. Carregamento dos Dados

Nesta etapa é realizado o carregamento da base analítica consolidada utilizada nos experimentos de modelagem estatística, inteligência artificial, sistemas de recomendação e análises avançadas de comportamento de consumo.

| Variável             | Tipo                | Descrição                             |
| -------------------- | ------------------- | ------------------------------------- |
| chave_anonimizada    | categórica          | Identificador único da nota fiscal    |
| data_hora            | datetime            | Data e hora da compra                 |
| mes_ano              | categórica temporal | Ano e mês da compra                   |
| dia_semana           | categórica          | Dia da semana da compra               |
| feriado              | binária             | Indica se a compra ocorreu em feriado |
| estacao_ano          | categórica          | Estação climática                     |
| temperatura_max      | numérica contínua   | Temperatura máxima do dia             |
| temperatura_min      | numérica contínua   | Temperatura mínima do dia             |
| temperatura_media    | numérica contínua   | Temperatura média do dia              |
| chuva_mm             | numérica contínua   | Volume de chuva                       |
| cat_temperatura      | categórica ordinal  | Faixa de temperatura                  |
| dia_chuvoso          | binária             | Indica ocorrência de chuva            |
| periodo_dia          | categórica ordinal  | Madrugada, manhã, tarde ou noite      |
| cnpj                 | categórica          | Identificador do supermercado         |
| supermercado         | categórica          | Nome do supermercado                  |
| qtd_total_nota       | numérica contínua   | Quantidade total de produtos da nota  |
| valor_total_nota     | numérica contínua   | Valor total da nota fiscal            |
| valor_total_tributos | numérica contínua   | Soma dos tributos da nota             |
| cod_produto          | categórica          | Código do produto                     |
| categoria_produto    | categórica          | Categoria do produto                  |
| produto              | textual/categórica  | Nome do produto                       |
| unidade              | categórica          | Unidade de venda                      |
| quantidade           | numérica contínua   | Quantidade comprada                   |
| preco_unitario       | numérica contínua   | Preço unitário                        |
| preco_total          | numérica contínua   | Valor total do item                   |

In [3]:
tic = time.time()

In [4]:
data_path = config.get(
    "paths.data"
)
df = pd.read_excel(data_path)
print(
    f"Linhas: {df.shape[0]}"
)
print(
    f"Colunas: {df.shape[1]}"
)
df.head()

Linhas: 3430
Colunas: 25


,chave_anonimizada,data_hora,mes_ano,dia_semana,feriado,estacao_ano,temperatura_max,temperatura_min,temperatura_media,chuva_mm,...,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,categoria_produto,produto,unidade,quantidade,preco_unitario,preco_total
0,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10176020001,CEREAL,CEREAL NESCAU 770G,UNIDADE,1.0,19.90,19.90
1,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,11421800001,MERCEARIA,BEB VERY 1250G SALAD,UNIDADE,1.0,7.70,7.70
2,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,18730001,MERCEARIA,ARR T JOAO LF T1 5kg,UNIDADE,1.0,25.90,25.90
3,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,65420001,BEBIDA NAO ALCOOLICA,BEB NESQUIK 200ML MO,UNIDADE,10.0,2.89,28.90
4,fcb669525bd643f21cb51d896c86adfe0933f5cb868f32...,2022-06-28 20:44:50,2022-06,TERCA,NAO,INVERNO,21.3,9.8,14.7,0.0,...,82,679.79,76.32,10566170001,MOLHO,EXT ELEF TP 140G,UNIDADE,2.0,2.59,5.18


In [5]:
toc = time.time()
print(f"\n\033[33mEtapa 1 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 1 | Tempo: (00:00:02) [19/05/2026 18:20:51 - [19/05/2026 18:20:53]


### 2. Entendimento Inicial dos Dados

**POSSÍVEIS VARIÁVEIS RESPOSTA - MODELOS SUPERVISIONADOS**

**Regressão**
| variável resposta | objetivo              |
| ----------------- | --------------------- |
| preco_unitario    | previsão de preço     |
| preco_total       | previsão de gasto     |
| valor_total_nota  | previsão de ticket    |
| quantidade        | previsão de consumo   |
| chuva_mm          | inferência contextual |

**Classificação**
| variável resposta | objetivo                                            |
| ----------------- | --------------------------------------------------- |
| feriado           | prever padrão feriado                               |
| categoria_produto | classificação textual                               |
| produto           | prever a compra de determinado produto (ex: Whisky) |

---

**MODELOS NÃO SUPERVISIONADOS**

**Clusterização**
| técnica | objetivo            |
| ------- | ------------------- |
| KMeans  | tipos de compra     |
| HDBSCAN | padrões raros       |
| PCA     | redução dimensional |
| UMAP    | embeddings visuais  |

---

**ASSOCIATION RULES**

| técnica   | objetivo                  |
| --------- | ------------------------- |
| FP-Growth | produtos comprados juntos |
| Apriori   | cesta                     |
| Lift      | relações fortes           |

---

**NLP / IA**

| técnica               | objetivo             |
| --------------------- | -------------------- |
| Embeddings            | similaridade         |
| LLM                   | insights automáticos |
| RAG                   | consulta inteligente |
| Recommendation System | recomendação         |

In [6]:
tic = time.time()

In [7]:
results = analyze_dataset(df)

2026-05-19 18:20:53,805 | src.core.model | INFO | Iniciando análise exploratória da base
2026-05-19 18:20:53,920 | src.core.model | INFO | Análise exploratória concluída


#### 2.1 Shape

In [8]:
results['shape']

{'linhas': 3430, 'colunas': 25}

#### 2.2 Nulos

In [9]:
results['missing']

,total_nulos,percentual_nulos
chave_anonimizada,0,0.0
data_hora,0,0.0
mes_ano,0,0.0
dia_semana,0,0.0
feriado,0,0.0
estacao_ano,0,0.0
temperatura_max,0,0.0
temperatura_min,0,0.0
temperatura_media,0,0.0
chuva_mm,0,0.0


#### 2.3 Estatísticas Numéricas

In [10]:
results['numeric_description']

,count,mean,std,min,25%,50%,75%,max
temperatura_max,3430.0,2.252845e+01,3.868437e+00,11.700,20.50,22.60,2.570000e+01,3.240000e+01
temperatura_min,3430.0,1.420184e+01,3.438475e+00,3.100,11.40,14.20,1.720000e+01,2.070000e+01
temperatura_media,3430.0,1.763583e+01,3.326277e+00,8.500,15.30,17.60,2.030000e+01,2.620000e+01
chuva_mm,3430.0,3.888047e+00,6.937823e+00,0.000,0.10,1.00,4.400000e+00,5.190000e+01
qtd_total_nota,3430.0,4.725539e+01,3.403687e+01,1.000,20.00,37.00,7.500000e+01,1.180000e+02
valor_total_nota,3430.0,3.889503e+02,2.884341e+02,3.490,152.68,284.20,6.483200e+02,9.504200e+02
valor_total_tributos,3430.0,1.049825e+02,9.329158e+01,0.000,21.23,83.53,1.693900e+02,3.234700e+02
cod_produto,3430.0,1.300014e+09,3.510617e+09,382.000,111165.00,265158.50,1.207733e+06,1.204739e+10
quantidade,3430.0,1.240356e+00,1.386944e+00,0.035,1.00,1.00,1.000000e+00,1.400000e+01
preco_unitario,3430.0,8.026592e+00,9.105268e+00,0.690,3.29,5.79,8.990000e+00,1.499000e+02


#### 2.4 Cardinalidade

In [11]:
results['cardinality']

,variavel,qtd_unicos
20,produto,1087
18,cod_produto,1087
24,preco_total,694
23,preco_unitario,445
22,quantidade,218
1,data_hora,177
0,chave_anonimizada,177
16,valor_total_nota,175
17,valor_total_tributos,169
6,temperatura_max,90


#### 2.5 Correlação

In [12]:
results['correlation']

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,preco_unitario,preco_total
temperatura_max,1.000000,0.739087,0.917545,-0.030700,-0.041186,-0.291535,-0.164825,-0.130256,0.070291,0.051188,0.023975,0.056574
temperatura_min,0.739087,1.000000,0.929979,0.294949,0.449478,-0.328289,-0.202540,-0.141422,-0.021087,0.059715,0.015666,0.052032
temperatura_media,0.917545,0.929979,1.000000,0.151388,0.233181,-0.366412,-0.225111,-0.175857,0.039449,0.056423,0.022911,0.059934
chuva_mm,-0.030700,0.294949,0.151388,1.000000,0.318444,-0.121475,-0.037022,-0.011028,-0.053504,0.042593,0.023496,0.047979
dia_chuvoso,-0.041186,0.449478,0.233181,0.318444,1.000000,-0.163560,-0.118743,-0.075763,-0.068258,0.032237,-0.020038,-0.003092
qtd_total_nota,-0.291535,-0.328289,-0.366412,-0.121475,-0.163560,1.000000,0.938853,0.890139,-0.194780,-0.035750,-0.019743,-0.032251
valor_total_nota,-0.164825,-0.202540,-0.225111,-0.037022,-0.118743,0.938853,1.000000,0.949038,-0.210496,0.001398,0.032688,0.041138
valor_total_tributos,-0.130256,-0.141422,-0.175857,-0.011028,-0.075763,0.890139,0.949038,1.000000,-0.321575,-0.006555,0.039041,0.046226
cod_produto,0.070291,-0.021087,0.039449,-0.053504,-0.068258,-0.194780,-0.210496,-0.321575,1.000000,0.060328,-0.024535,-0.004196
quantidade,0.051188,0.059715,0.056423,0.042593,0.032237,-0.035750,0.001398,-0.006555,0.060328,1.000000,-0.115693,0.236293


#### 2.7 Outliers

In [13]:
pd.DataFrame(results['outliers']).T

,total_outliers,percentual_outliers
temperatura_max,20.0,0.58
temperatura_min,0.0,0.00
temperatura_media,0.0,0.00
chuva_mm,416.0,12.13
qtd_total_nota,0.0,0.00
valor_total_nota,0.0,0.00
valor_total_tributos,0.0,0.00
cod_produto,784.0,22.86
quantidade,531.0,15.48
preco_unitario,293.0,8.54


#### 2.8 Categorias mais frequentes

In [14]:
results['categorical_summary']['categoria_produto']

categoria_produto
BEBIDA NAO ALCOOLICA    1178
MERCEARIA                395
PADARIA E FRIOS          240
MASSA                    212
CONGELADO                206
BOMBONIERE               192
HIGIENE E LIMPEZA        166
BEBIDA ALCOOLICA         151
MOLHO                    136
HORTIFRUTI               114
Name: count, dtype: int64

#### 2.9 Intervalo Temporal

In [15]:
results['date_range']

{'data_min': Timestamp('2022-06-28 20:44:50'),
 'data_max': Timestamp('2026-05-17 19:32:05'),
 'dias': 1418}

In [16]:
toc = time.time()
print(f"\n\033[33mEtapa 2 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 2 | Tempo: (00:00:00) [19/05/2026 18:20:53 - [19/05/2026 18:20:54]


### 3. Feature Engineering

In [17]:
tic = time.time()

In [18]:
# ==========================================================
# FEATURE ENGINEERING
# ==========================================================

df_model = feature_engineering(df)

print(
    f'Colunas originais: {df.shape[1]}'
)

print(
    f'Colunas após FE: {df_model.shape[1]}'
)

2026-05-19 18:20:54,194 | src.core.model | INFO | Iniciando feature engineering
2026-05-19 18:20:54,206 | src.core.model | INFO | Criando features temporais
2026-05-19 18:20:54,223 | src.core.model | INFO | Criando features financeiras
2026-05-19 18:20:54,235 | src.core.model | INFO | Criando features de frequência
2026-05-19 18:20:54,254 | src.core.model | INFO | Criando features de preço
2026-05-19 18:20:54,277 | src.core.model | INFO | Criando features por nota fiscal
2026-05-19 18:20:54,300 | src.core.model | INFO | Criando features climáticas
2026-05-19 18:20:54,307 | src.core.model | INFO | Criando variáveis normalizadas
2026-05-19 18:20:54,321 | src.core.model | INFO | Criando features textuais
2026-05-19 18:20:54,338 | src.core.model | INFO | Criando flags analíticas
2026-05-19 18:20:54,347 | src.core.model | INFO | Feature engineering concluído | Novas colunas: 31


Colunas originais: 25
Colunas após FE: 56


#### 3.1 Novas Variáveis

In [19]:
df_model[[
    'produto',
    'produto_clean',
    'ticket_medio_item',
    'percentual_tributo',
    'produto_promocao',
    'qtd_itens_nota'
]].head()

,produto,produto_clean,ticket_medio_item,percentual_tributo,produto_promocao,qtd_itens_nota
0,CEREAL NESCAU 770G,CEREAL NESCAU 770G,8.290122,0.11227,0,82
1,BEB VERY 1250G SALAD,BEB VERY 1250G SALAD,8.290122,0.11227,0,82
2,ARR T JOAO LF T1 5kg,ARR T JOAO LF T1 5KG,8.290122,0.11227,0,82
3,BEB NESQUIK 200ML MO,BEB NESQUIK 200ML MO,8.290122,0.11227,0,82
4,EXT ELEF TP 140G,EXT ELEF TP 140G,8.290122,0.11227,0,82


#### 3.2 Sample

In [20]:
novas_colunas = [

    col for col in df_model.columns

    if col not in df.columns
]

novas_colunas

['ano',
 'mes',
 'dia',
 'hora',
 'dia_ano',
 'semana_ano',
 'fim_semana',
 'ticket_medio_item',
 'percentual_tributo',
 'preco_por_quantidade',
 'frequencia_produto',
 'frequencia_categoria',
 'frequencia_supermercado',
 'desvio_preco_produto',
 'percentual_desvio_preco',
 'produto_promocao',
 'qtd_itens_nota',
 'qtd_categorias_nota',
 'amplitude_termica',
 'chuva_intensa',
 'preco_unitario_scaled',
 'preco_total_scaled',
 'quantidade_scaled',
 'valor_total_nota_scaled',
 'temperatura_media_scaled',
 'chuva_mm_scaled',
 'produto_clean',
 'produto_tamanho_texto',
 'compra_grande',
 'produto_caro',
 'produto_barato']

#### 3.3 Promoções Detectadas

In [21]:
df_model[
    df_model['produto_promocao'] == 1
][[
    'produto',
    'preco_unitario',
    'percentual_desvio_preco'
]].head(20)

,produto,preco_unitario,percentual_desvio_preco
13,FANDANGOS 45G QJO,2.69,-0.154088
14,MILHO QUERO 170G,3.39,-0.168438
15,FANDANGOS 45G QJO,2.69,-0.154088
53,MILHO QUERO 170G,3.39,-0.168438
59,POPCOR YOKI 100G BAC,3.98,-0.168234
86,BEB NESQUIK 200ML MO,2.29,-0.276207
115,MORANGO FRES 250G,7.89,-0.160192
128,LIMAO THAITI kg,1.99,-0.412979
130,TOMATE ITALIANO kg,2.99,-0.505422
136,FANDANGOS 45G QJO,2.69,-0.154088


#### 3.4 Correlação Novas

In [22]:
df_model.corr(
    numeric_only=True
)

,temperatura_max,temperatura_min,temperatura_media,chuva_mm,dia_chuvoso,qtd_total_nota,valor_total_nota,valor_total_tributos,cod_produto,quantidade,...,preco_unitario_scaled,preco_total_scaled,quantidade_scaled,valor_total_nota_scaled,temperatura_media_scaled,chuva_mm_scaled,produto_tamanho_texto,compra_grande,produto_caro,produto_barato
temperatura_max,1.000000,0.739087,0.917545,-0.030700,-0.041186,-0.291535,-0.164825,-0.130256,7.029102e-02,0.051188,...,0.023975,0.056574,0.051188,-0.164825,0.917545,-0.030700,4.164396e-02,-0.047698,0.000906,0.019433
temperatura_min,0.739087,1.000000,0.929979,0.294949,0.449478,-0.328289,-0.202540,-0.141422,-2.108657e-02,0.059715,...,0.015666,0.052032,0.059715,-0.202540,0.929979,0.294949,1.574882e-02,-0.125082,0.014894,0.039672
temperatura_media,0.917545,0.929979,1.000000,0.151388,0.233181,-0.366412,-0.225111,-0.175857,3.944902e-02,0.056423,...,0.022911,0.059934,0.056423,-0.225111,1.000000,0.151388,4.052453e-02,-0.140239,0.010892,0.033581
chuva_mm,-0.030700,0.294949,0.151388,1.000000,0.318444,-0.121475,-0.037022,-0.011028,-5.350415e-02,0.042593,...,0.023496,0.047979,0.042593,-0.037022,0.151388,1.000000,-3.054419e-02,0.095916,0.037931,0.015382
dia_chuvoso,-0.041186,0.449478,0.233181,0.318444,1.000000,-0.163560,-0.118743,-0.075763,-6.825754e-02,0.032237,...,-0.020038,-0.003092,0.032237,-0.118743,0.233181,0.318444,-6.583357e-02,-0.286104,0.008275,0.050666
qtd_total_nota,-0.291535,-0.328289,-0.366412,-0.121475,-0.163560,1.000000,0.938853,0.890139,-1.947799e-01,-0.035750,...,-0.019743,-0.032251,-0.035750,0.938853,-0.366412,-0.121475,-1.661119e-01,0.550745,-0.002177,-0.018567
valor_total_nota,-0.164825,-0.202540,-0.225111,-0.037022,-0.118743,0.938853,1.000000,0.949038,-2.104958e-01,0.001398,...,0.032688,0.041138,0.001398,1.000000,-0.225111,-0.037022,-1.616585e-01,0.626752,0.038607,-0.035992
valor_total_tributos,-0.130256,-0.141422,-0.175857,-0.011028,-0.075763,0.890139,0.949038,1.000000,-3.215752e-01,-0.006555,...,0.039041,0.046226,-0.006555,0.949038,-0.175857,-0.011028,-2.040146e-01,0.631874,0.031821,-0.017756
cod_produto,0.070291,-0.021087,0.039449,-0.053504,-0.068258,-0.194780,-0.210496,-0.321575,1.000000e+00,0.060328,...,-0.024535,-0.004196,0.060328,-0.210496,0.039449,-0.053504,7.640832e-02,-0.131675,-0.007349,0.020100
quantidade,0.051188,0.059715,0.056423,0.042593,0.032237,-0.035750,0.001398,-0.006555,6.032833e-02,1.000000,...,-0.115693,0.236293,1.000000,0.001398,0.056423,0.042593,8.558511e-02,-0.004176,-0.080739,0.223979


In [23]:
toc = time.time()
print(f"\n\033[33mEtapa 3 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 3 | Tempo: (00:00:00) [19/05/2026 18:20:54 - [19/05/2026 18:20:54]


### 4. Market Basket Analysis

Nesta etapa são identificadas relações entre produtos e categorias frequentemente comprados em conjunto utilizando técnicas de associação.

In [24]:
tic = time.time()

In [25]:
# ==========================================================
# MARKET BASKET ANALYSIS
# ==========================================================

basket_results = run_market_basket_analysis(

    df=df_model,

    min_support=0.01,

    metric='lift',

    min_threshold=1.2,

    max_len=3,

    top_n_rules=20
)

2026-05-19 18:20:54,693 | src.core.model | INFO | Iniciando Market Basket Analysis
2026-05-19 18:20:54,707 | src.core.model | INFO | Realizando limpeza dos dados
2026-05-19 18:20:54,713 | src.core.model | INFO | Construindo basket matrix
2026-05-19 18:20:54,728 | src.core.model | INFO | Executando FP-Growth
2026-05-19 18:20:55,789 | src.core.model | INFO | Gerando regras de associação
2026-05-19 18:20:59,345 | src.core.model | INFO | Calculando frequência dos produtos
2026-05-19 18:20:59,349 | src.core.model | INFO | Calculando métricas resumidas
2026-05-19 18:20:59,358 | src.core.model | INFO | Market Basket Analysis concluído


#### 4.1 Resumo Geral

In [26]:
basket_results['metrics_summary']

{'total_notas': 177,
 'total_produtos': 1085,
 'total_regras': 297410,
 'media_lift': np.float64(14.9863),
 'media_confidence': np.float64(0.4971),
 'media_support': np.float64(0.0132)}

#### 4.2 Regras Mais Fortes

In [27]:
basket_results['top_rules'][[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]]

,antecedents,consequents,support,confidence,lift
0,"FILE SEARA PEITO 1KG, CAFE MELITTA 500G",TEMP KININO 30G,0.011299,1.0,88.5
1,TEMP KININO 30G,"FILE SEARA PEITO 1KG, CAFE MELITTA 500G",0.011299,1.0,88.5
2,"FILE SEARA PEITO 1KG, AMAC DOWNY 500ML",TEMP KININO 30G,0.011299,1.0,88.5
3,TEMP KININO 30G,"FILE SEARA PEITO 1KG, AMAC DOWNY 500ML",0.011299,1.0,88.5
4,"FILE SEARA PEITO 1KG, LASANH PERDIGAO 600G",TEMP KININO 30G,0.011299,1.0,88.5
5,TEMP KININO 30G,"FILE SEARA PEITO 1KG, LASANH PERDIGAO 600G",0.011299,1.0,88.5
6,"FILE SEARA PEITO 1KG, PAO VISCONTI 400G",TEMP KININO 30G,0.011299,1.0,88.5
7,TEMP KININO 30G,"FILE SEARA PEITO 1KG, PAO VISCONTI 400G",0.011299,1.0,88.5
8,"FILE SEARA PEITO 1KG, SALSINHA CAVALLI 50G",TEMP KININO 30G,0.011299,1.0,88.5
9,TEMP KININO 30G,"FILE SEARA PEITO 1KG, SALSINHA CAVALLI 50G",0.011299,1.0,88.5


#### 4.3 Itemsets Frequentes

In [28]:
basket_results['frequent_items']

,support,itemsets
0,0.158192,frozenset({ENERG MONST 473ML})
1,0.129944,frozenset({WHISKY PASSP 670ML})
2,0.129944,frozenset({DROPS HALLS 28G})
3,0.129944,frozenset({SUCO MAIS MARACUJ 1L})
4,0.124294,frozenset({LIMAO KG})
...,...,...
54123,0.011299,"frozenset({MACA FUJI KG, BATATA MONALISA KG})"
54124,0.011299,"frozenset({MACA FUJI KG, PRESUNTO SEARA 200G, ..."
54125,0.011299,"frozenset({MACA FUJI KG, PRESUNTO SEARA 200G, ..."
54126,0.011299,"frozenset({MACA FUJI KG, EXT TOM ELEFANT 300G,..."


#### 4.4 Produtos Mais Frequentes

In [29]:
basket_results['product_frequency'].head(20)

,produto,frequencia
0,BEB LAC YOPRO 250ML,52
1,SUCO MAIS MARACUJ 1L,46
2,ENERG MONST 473ML,45
3,BEB LAC DANONE 250ML,45
4,AGUA COCO SOCO 200ML,42
5,DROPS HALLS 28G,39
6,MAC NISSIN 99G,39
7,BEB NESQUIK 200ML MO,36
8,LEITE TIROL TP 1L,35
9,CREME LTE LIDER 200G,35


#### 4.5 Regras Muito Fortes

In [30]:
basket_results['rules'][
    basket_results['rules']['lift'] >= 3
][[
    'antecedents',
    'consequents',
    'lift',
    'confidence'
]]

,antecedents,consequents,lift,confidence
0,"FILE SEARA PEITO 1KG, CAFE MELITTA 500G",TEMP KININO 30G,88.500000,1.000000
1,TEMP KININO 30G,"FILE SEARA PEITO 1KG, CAFE MELITTA 500G",88.500000,1.000000
2,"FILE SEARA PEITO 1KG, AMAC DOWNY 500ML",TEMP KININO 30G,88.500000,1.000000
3,TEMP KININO 30G,"FILE SEARA PEITO 1KG, AMAC DOWNY 500ML",88.500000,1.000000
4,"FILE SEARA PEITO 1KG, LASANH PERDIGAO 600G",TEMP KININO 30G,88.500000,1.000000
...,...,...,...,...
293947,CEBOLA KG,"TEKITOS SEARA 300G, BANANA CATURRA KG",3.017045,0.136364
293948,CEBOLA KG,SABAO OMO 1 6KG,3.017045,0.136364
293949,LIMAO KG,FILE FGO SEARA KG,3.017045,0.136364
293950,CEBOLA KG,QUEIJO KOLLAC 160G,3.017045,0.136364


In [31]:
toc = time.time()
print(f"\n\033[33mEtapa 4 | Tempo:\033[0;0m {tictoc(tic, toc)}")


Etapa 4 | Tempo: (00:00:05) [19/05/2026 18:20:54 - [19/05/2026 18:20:59]


### 5. NLP & Embeddings

Nesta etapa são gerados embeddings semânticos dos produtos para análises de similaridade, agrupamento e recomendação.

In [ ]:
tic = time.time()

In [ ]:
df['produto_clean']

In [ ]:
!pip install sentence_transformers

In [ ]:
from sentence_transformers import (
    SentenceTransformer
)

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

In [ ]:
embeddings = model.encode(
    df['produto'].unique()
)

In [ ]:
cosine_similarity(...)

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 5 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 6. IA Generativa e Insights Automáticos

Nesta etapa são exploradas abordagens de LLMs para geração automática de insights, explicações estatísticas e interpretação dos padrões de consumo.

In [ ]:
tic = time.time()

In [ ]:
def generate_insight(...):

In [ ]:
toc = time.time()
print(f"\n\033[33mEtapa 6 | Tempo:\033[0;0m {tictoc(tic, toc)}")

### 8. Próximos Passos

- Sistemas de recomendação
- Forecast temporal
- Detecção de anomalias
- RAG sobre dados financeiros
- IA contextual com clima e inflação

## 3. Tempo Decorrido

In [ ]:
toc_geral = time.time()
print(f"\n\033[33mTempo decorrido no notebook:\033[0;0m {tictoc(tic_geral, toc_geral)}")